# ACORN prediction and SHAP attribution

Runs a prediction — live or for a chosen past time — and explains it: which
solar wind driver, at which lag, produced the current in the selected area.

Set `WHEN` and `MODE` in the next cell and run the whole notebook. Cells
outside the selected mode skip themselves.

`WHEN` is either `"realtime"` or a timestamp string. For a past time the sci
model is used when its inputs are all available for that window, and the op
model otherwise — sci needs SYM_H/ASY_H and SuperMAG SML/SMU, which exist in
the OMNI archive but not in the live feed.

| MODE | what runs |
|---|---|
| `"overall"` | attribution over the whole grid |
| `"regions"` | all twelve evaluation regions, one shared data fetch |
| `"custom"` | one arbitrary MLAT/MLT box |

In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

SRC = Path("..")
sys.path.insert(0, str(SRC.resolve()))
os.environ.setdefault("SUPERMAG_USERID", "acorn_user")

import utils, inference

# ── when to predict ──────────────────────────────────────────────────────
WHEN = "realtime"                    # "realtime", or e.g. "2023-05-06 05:00:00"

# ── what to explain ──────────────────────────────────────────────────────
MODE = "overall"          # "overall" | "regions" | "custom"

MLAT_RANGE = (65, 75)     # used when MODE == "custom"
MLT_RANGE  = (21, 2)      # used when MODE == "custom"; may wrap midnight

CHANNEL    = 0            # 0 = mean field, 1 = predicted std
TOP_N      = 4            # drivers shown in the lag-profile panels
# ─────────────────────────────────────────────────────────────────────────

assert MODE in ("overall", "regions", "custom"), MODE
REALTIME  = str(WHEN).lower() == "realtime"
TIMESTAMP = None if REALTIME else str(WHEN)

print(f"WHEN = {'real-time' if REALTIME else TIMESTAMP}")
print(f"MODE = {MODE}")
if MODE == "custom":
    print(f"  MLAT {MLAT_RANGE}, MLT {MLT_RANGE}")

## 1. Prediction

In [ ]:
fac, variant = inference.auto_inference(
    realtime=REALTIME, timestamp=TIMESTAMP)

mean, std, t, sequences = fac.predict(timestamp=TIMESTAMP)

m = mean[0] if mean.ndim == 3 else mean
s = std[0]  if std.ndim  == 3 else std

print(f"\nmodel: {variant}   timestamp: {t}")
print(f"mean {m.shape}  {m.min():.3f} .. {m.max():.3f}")
print(f"std  {s.shape}  {s.min():.3f} .. {s.max():.3f}")

inference.testing_polar_plot(
    [m, s], t,
    [f"ACORN {variant.title()} mean", f"ACORN {variant.title()} std"])

## 2. Attribution

The baseline is the training climatology, sampled across activity deciles, so
values read as "what makes this moment unusual relative to the conditions the
model learned". The first call builds and caches that background.

In `regions` mode all twelve are attributed from a single data fetch and a
single background, so they are directly comparable.

In [ ]:
results = None
expl = None

if MODE == "regions":
    results = fac.explain_regions(timestamp=TIMESTAMP, channel=CHANNEL)
else:
    target = "overall" if MODE == "overall" else "custom"
    expl = fac.explain(
        target=target,
        timestamp=TIMESTAMP,
        mlat_range=None if MODE == "overall" else MLAT_RANGE,
        mlt_range=None if MODE == "overall" else MLT_RANGE,
        channel=CHANNEL)
    print(f"\nregion:     {expl['label']}")
    print(f"baseline:   {expl['baseline']}")
    print(f"prediction: {expl['prediction']:.4f}")
    print(f"climatology: {expl['base_value']:.4f}")
    print(f"departure:  {expl['prediction'] - expl['base_value']:+.4f}")

### Area covered

In [ ]:
if MODE == "regions":
    inference.plot_all_shap_regions(field=m)
else:
    inference.plot_shap_region(expl, field=m)

## 3. Single-area breakdown

Runs in `overall` and `custom` modes.

In [ ]:
if expl is None:
    print("MODE = 'regions' — see section 4.")
else:
    sgn   = expl["param_signed"]
    order = np.argsort(np.abs(sgn))[::-1]
    names = np.array(expl["params"])[order]
    vals_s, pcts = sgn[order], expl["signed_pct"][order]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(len(names)), vals_s,
            color=["firebrick" if v >= 0 else "steelblue" for v in vals_s])
    ax.axvline(0, color="k", lw=0.8)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel("net SHAP  (red raises |FAC|, blue lowers)")
    ax.set_title(f"{expl['label']} — {expl['timestamp']}")
    span = np.abs(vals_s).max()
    for i, (v, p) in enumerate(zip(vals_s, pcts)):
        ax.text(v + 0.02 * span * (1 if v >= 0 else -1), i, f"{p:.1f}%",
                va="center", fontsize=9, ha="left" if v >= 0 else "right")
    ax.set_xlim(-span * 1.25, span * 1.25)
    plt.tight_layout(); plt.show()

In [ ]:
if expl is not None:
    summary = pd.DataFrame({
        "net SHAP": expl["param_signed"],
        "net %":    expl["signed_pct"],
        "|SHAP| %": expl["param_pct"],
        "cancel":   expl["cancellation"],
        "peak lag": expl["peak_lag"],
    }, index=expl["params"])
    if expl["physical"] is not None:
        summary["value now"]  = expl["physical"][-1]
        summary["window mean"] = expl["physical"].mean(axis=0)
    display(summary.reindex(summary["net SHAP"].abs()
                            .sort_values(ascending=False).index).round(2))

`cancel` is the signed sum over the absolute sum: near ±1 means the driver
pushed consistently one way across the lookback, near 0 means it reversed and
cancelled itself.

### SHAP by lag and driver

In [ ]:
if expl is not None:
    v = expl["shap"]
    lim = np.abs(v).max()
    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.pcolormesh(np.arange(v.shape[1] + 1), np.arange(v.shape[0] + 1),
                       v, cmap="bwr", vmin=-lim, vmax=lim)
    ax.set_xticks(np.arange(v.shape[1]) + 0.5)
    ax.set_xticklabels(expl["params"], rotation=45, ha="right")
    ax.set_ylabel("lag (minutes before prediction)")
    ax.set_title(f"SHAP by lag and driver — {expl['label']}")
    fig.colorbar(im, ax=ax).set_label("SHAP value")
    plt.tight_layout(); plt.show()

### Lag profile with driver values

SHAP and the driver's physical value as two lines on shared time axes.

In [ ]:
if expl is not None:
    top = np.argsort(np.abs(expl["param_signed"]))[::-1][:TOP_N]
    lag_axis = np.arange(expl["shap"].shape[0])[::-1]   # 0 = most recent

    fig, axes = plt.subplots(TOP_N, 1, figsize=(10, 2.4 * TOP_N), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, top):
        ax.plot(lag_axis, expl["shap"][:, i], color="crimson", lw=1.6,
                label="SHAP")
        ax.axhline(0, color="k", lw=0.7)
        ax.set_ylabel("SHAP", color="crimson", fontsize=9)
        ax.tick_params(axis="y", labelcolor="crimson", labelsize=8)
        ax.grid(alpha=0.25)
        if expl["physical"] is not None:
            ax2 = ax.twinx()
            ax2.plot(lag_axis, expl["physical"][:, i], color="k", lw=1.4,
                     label="value")
            ax2.set_ylabel(expl["params"][i], fontsize=9)
            ax2.tick_params(axis="y", labelsize=8)
    axes[-1].set_xlabel("minutes before prediction")
    axes[-1].invert_xaxis()
    plt.suptitle(f"{expl['label']} — SHAP (red) and driver value (black)")
    plt.tight_layout(); plt.show()

## 4. All regions

Runs in `regions` mode only.

### Net SHAP by region, one panel per driver

Each panel shows where in the polar grid that driver pushed the prediction up
or down. Colour scales are per panel, so read within a panel rather than
across.

In [ ]:
if results is None:
    print("MODE is not 'regions' — see section 3.")
else:
    inference.plot_region_shap_polar(results)

### SHAP by lag and driver, per region

In [ ]:
if results is not None:
    inference.plot_region_lag_heatmaps(results)

### Net contribution table

In [ ]:
if results is not None:
    table = pd.DataFrame(
        {lab: pd.Series(e["param_signed"], index=e["params"])
         for lab, e in results.items()}).T
    display(table.round(3))

In [ ]:
if results is not None:
    lim = np.abs(table.values).max()
    fig, ax = plt.subplots(figsize=(11, 7))
    im = ax.pcolormesh(table.values, cmap="bwr", vmin=-lim, vmax=lim)
    ax.set_xticks(np.arange(len(table.columns)) + 0.5)
    ax.set_xticklabels(table.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(table.index)) + 0.5)
    ax.set_yticklabels(table.index)
    ax.set_title("Net SHAP contribution by region")
    fig.colorbar(im, ax=ax).set_label("net SHAP")
    plt.tight_layout(); plt.show()

### Per-region driver ranking

In [ ]:
if results is not None:
    for lab, e in results.items():
        order = np.argsort(np.abs(e["param_signed"]))[::-1][:5]
        top_str = ", ".join(
            f"{e['params'][i]} {e['param_signed'][i]:+.3f}" for i in order)
        print(f"{lab:<16} {top_str}")